In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/loan-approval-systemlas/clientes.csv


In [2]:
import pandas as pd

import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.impute import SimpleImputer 
df = pd.read_csv("/kaggle/input/loan-approval-systemlas/clientes.csv")

df = df.drop('cod_cliente', axis=1)
 
cat_columns = ['sexo', 'estado_civil', 'dependentes', 'empregado', 'renda_conjuge', 'imovel']
for col in cat_columns:

    df[col].fillna(df[col].mode()[0], inplace=True)
num_columns = ['emprestimo', 'prestacao_mensal', 'historico_credito']

for col in num_columns:

    df[col].fillna(df[col].mean(), inplace=True)
 
label_encoder = LabelEncoder()

for col in cat_columns:

    df[col] = label_encoder.fit_transform(df[col])
 

df['aprovacao_emprestimo'] = df['aprovacao_emprestimo'].apply(lambda x: 1 if x == 'Y' else 0)
 

print(df.isnull().sum())
 

sexo                    0
estado_civil            0
dependentes             0
educacao                0
empregado               0
renda                   0
renda_conjuge           0
emprestimo              0
prestacao_mensal        0
historico_credito       0
imovel                  0
aprovacao_emprestimo    0
dtype: int64


<ipython-input-2-0e06c50569bc>:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
<ipython-input-2-0e06c50569bc>:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [4]:
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# Separate features and target
X = df.drop(columns=["aprovacao_emprestimo"])
y = df["aprovacao_emprestimo"]

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
pca = PCA(n_components=5)  # Adjust components if needed
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Apply LDA
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)


In [9]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"{model.__class__.__name__}: Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={recall:.4f}, F1-Score={f1:.4f}")

In [10]:
log_reg = LogisticRegression()
evaluate_model(log_reg, X_train_pca, X_test_pca, y_train, y_test)

evaluate_model(log_reg, X_train_lda, X_test_lda, y_train, y_test)


LogisticRegression: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
LogisticRegression: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587


In [11]:

# Random Forest
rf = RandomForestClassifier()
evaluate_model(rf, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(rf, X_train_lda, X_test_lda, y_train, y_test)

RandomForestClassifier: Accuracy=0.7398, Precision=0.7449, Recall=0.9125, F1-Score=0.8202
RandomForestClassifier: Accuracy=0.7317, Precision=0.7701, Recall=0.8375, F1-Score=0.8024


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [13]:
# Support Vector Machine (SVM)
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
evaluate_model(svm, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(svm, X_train_lda, X_test_lda, y_train, y_test)

# Gradient Boosting Classifier
gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
evaluate_model(gbc, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(gbc, X_train_lda, X_test_lda, y_train, y_test)

SVC: Accuracy=0.7886, Precision=0.7547, Recall=1.0000, F1-Score=0.8602
SVC: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
GradientBoostingClassifier: Accuracy=0.7317, Precision=0.7474, Recall=0.8875, F1-Score=0.8114
GradientBoostingClassifier: Accuracy=0.7236, Precision=0.7396, Recall=0.8875, F1-Score=0.8068


In [14]:
# Optimize Support Vector Machine (SVM) with GridSearchCV
from sklearn.model_selection import GridSearchCV

svm_params = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.01, 0.1], 'kernel': ['rbf', 'poly']}
gs_svm = GridSearchCV(SVC(random_state=42), svm_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_svm.fit(X_train_pca, y_train)
best_svm = gs_svm.best_estimator_
evaluate_model(best_svm, X_train_pca, X_test_pca, y_train, y_test)

gs_svm.fit(X_train_lda, y_train)
best_svm = gs_svm.best_estimator_
evaluate_model(best_svm, X_train_lda, X_test_lda, y_train, y_test)

# Gradient Boosting Classifier
gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
evaluate_model(gbc, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(gbc, X_train_lda, X_test_lda, y_train, y_test)


SVC: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
SVC: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
GradientBoostingClassifier: Accuracy=0.7317, Precision=0.7474, Recall=0.8875, F1-Score=0.8114
GradientBoostingClassifier: Accuracy=0.7236, Precision=0.7396, Recall=0.8875, F1-Score=0.8068


In [15]:
# Optimize Random Forest using GridSearchCV
rf_params = {'n_estimators': [200, 300, 500], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train_pca, y_train)
best_rf = gs_rf.best_estimator_
evaluate_model(best_rf, X_train_pca, X_test_pca, y_train, y_test)

gs_rf.fit(X_train_lda, y_train)
best_rf = gs_rf.best_estimator_
evaluate_model(best_rf, X_train_lda, X_test_lda, y_train, y_test)

# Optimize Support Vector Machine (SVM) with GridSearchCV
svm_params = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.01, 0.1], 'kernel': ['rbf', 'poly']}
gs_svm = GridSearchCV(SVC(random_state=42), svm_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_svm.fit(X_train_pca, y_train)
best_svm = gs_svm.best_estimator_
evaluate_model(best_svm, X_train_pca, X_test_pca, y_train, y_test)

gs_svm.fit(X_train_lda, y_train)
best_svm = gs_svm.best_estimator_
evaluate_model(best_svm, X_train_lda, X_test_lda, y_train, y_test)

# Gradient Boosting Classifier
gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
evaluate_model(gbc, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(gbc, X_train_lda, X_test_lda, y_train, y_test)


RandomForestClassifier: Accuracy=0.7642, Precision=0.7525, Recall=0.9500, F1-Score=0.8398
RandomForestClassifier: Accuracy=0.7805, Precision=0.7573, Recall=0.9750, F1-Score=0.8525
SVC: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
SVC: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587
GradientBoostingClassifier: Accuracy=0.7317, Precision=0.7474, Recall=0.8875, F1-Score=0.8114
GradientBoostingClassifier: Accuracy=0.7236, Precision=0.7396, Recall=0.8875, F1-Score=0.8068


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [17]:
# Gradient Boosting Classifier
gbc = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
evaluate_model(gbc, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(gbc, X_train_lda, X_test_lda, y_train, y_test)

# Ensemble Learning - Voting Classifier
voting_clf = VotingClassifier(estimators=[('rf', best_rf), ('svm', best_svm), ('gbc', gbc)], voting='hard')
evaluate_model(voting_clf, X_train_pca, X_test_pca, y_train, y_test)
evaluate_model(voting_clf, X_train_lda, X_test_lda, y_train, y_test)

GradientBoostingClassifier: Accuracy=0.7317, Precision=0.7423, Recall=0.9000, F1-Score=0.8136
GradientBoostingClassifier: Accuracy=0.7236, Precision=0.7614, Recall=0.8375, F1-Score=0.7976
VotingClassifier: Accuracy=0.7724, Precision=0.7549, Recall=0.9625, F1-Score=0.8462
VotingClassifier: Accuracy=0.7886, Precision=0.7596, Recall=0.9875, F1-Score=0.8587


In [18]:
# Train and evaluate models using Stratified K-Fold Cross Validation
def evaluate_model_cv(model, X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)  # Increased folds to 10 for better generalization
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    print(f"{model.__class__.__name__}: Mean Accuracy={scores.mean():.4f}, Std Dev={scores.std():.4f}")

# Optimize Random Forest using GridSearchCV
rf_params = {'n_estimators': [500, 700], 'max_depth': [30, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42, class_weight='balanced', bootstrap=True), rf_params, cv=10, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train_pca, y_train)
best_rf = gs_rf.best_estimator_
evaluate_model_cv(best_rf, X_train_pca, y_train)

# Optimize Support Vector Machine (SVM) with GridSearchCV
svm_params = {'C': [1, 10, 50], 'gamma': ['scale', 0.01, 0.001], 'kernel': ['rbf']}
gs_svm = GridSearchCV(SVC(random_state=42, class_weight='balanced'), svm_params, cv=10, scoring='accuracy', n_jobs=-1)
gs_svm.fit(X_train_pca, y_train)
best_svm = gs_svm.best_estimator_
evaluate_model_cv(best_svm, X_train_pca, y_train)

# Gradient Boosting Classifier
gbc = GradientBoostingClassifier(n_estimators=500, learning_rate=0.03, max_depth=9, random_state=42)  # Tuned parameters
evaluate_model_cv(gbc, X_train_pca, y_train)

# Ensemble Learning - Voting Classifier
voting_clf = VotingClassifier(estimators=[('rf', best_rf), ('svm', best_svm), ('gbc', gbc)], voting='hard')
evaluate_model_cv(voting_clf, X_train_pca, y_train)


RandomForestClassifier: Mean Accuracy=0.7904, Std Dev=0.0361
SVC: Mean Accuracy=0.8046, Std Dev=0.0252
GradientBoostingClassifier: Mean Accuracy=0.7679, Std Dev=0.0351
VotingClassifier: Mean Accuracy=0.7883, Std Dev=0.0292


In [19]:
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Define hyperparameter grid for SVC
param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto", 0.01, 0.1, 1],
    "kernel": ["rbf", "linear", "poly", "sigmoid"]
}

# Perform GridSearchCV on SVC
svc = SVC()
grid_search = GridSearchCV(svc, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get best model
best_svc = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Evaluate the best model on test data
y_pred = best_svc.predict(X_test)

results = {
    "Best Parameters": best_params,
    "Train Accuracy": best_score,
    "Test Accuracy": accuracy_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred)
}

print(results)

{'Best Parameters': {'C': 10, 'gamma': 1, 'kernel': 'rbf'}, 'Train Accuracy': 0.8133333333333332, 'Test Accuracy': 0.7751479289940828, 'F1 Score': 0.7564102564102564, 'Precision': 0.7283950617283951, 'Recall': 0.7866666666666666}
